# Basic workflow
## Obtain Data from API
Paste your token in the token variable

In [1]:
import requests

api_url='https://xray-api.renecol.org'
token='IYFMLoaMyEBBaqin7jPrlfJlZa503b'
image_suffix='/api/imagedata/'
response = requests.get(headers={'Authorization': 'Bearer ' + token},url=api_url+image_suffix).json()
print(response)

[{'id': 293, 'upload_user': {'id': 10, 'first_name': 'Luisa', 'last_name': 'Thomas', 'email': 'luisalulo@web.de', 'username': 'luisalulo@web.de'}, 'measure_dataset': [], 'referring_patient': {'id': 41, 'hasClinicalBaseData': 1, 'completeNessClinicalBaseData': 33, 'numberClinicalEntries': 0, 'completeNessClinicalEntries': 0, 'hasICFxEntry': '', 'progresstag': [{'id': 3, 'owner': {'id': 4, 'first_name': 'Peter', 'last_name': 'Bernstein', 'email': 'bernspe@gmail.com', 'username': 'bernspe', 'institution': {'id': 1, 'uuid': '13bc7091-d8b1-4bfa-978e-2dd7dcef10d0', 'logo_url': 'https://www.asklepios.com/.resources/asklepios-theme/ccefa24/img/logo-asklepios.svg', 'name': 'Asklepios Hohwald WSZ', 'created_at': '2025-09-30T09:13:47.675742Z'}}, 'last_modified': '2025-11-10T14:19:43.843297Z', 'status': 'START', 'msg': None}], 'uuid': 'c6255376-6caa-4b78-80e2-141aafb1d60b', 'patient_id': '2590', 'created_at': '2025-11-07T11:42:10.513655Z', 'institution': 1}, 'patient_id_integrity_check': True, 'uu

In [2]:
import pandas as pd
df = pd.DataFrame(response)

df2 = df[['id','patient_id','measure_dataset','referring_patient','img_type']]
# filter those rows where measure_dataset is not an empty list
df3 = df2[df2['measure_dataset'].map(lambda d: len(d)) > 0]

# unpack measure_dataset column and combine it with the rest of df3, renaming the columns appropriately 
df4 = pd.concat([df3.drop(columns=['measure_dataset']), df3['measure_dataset'].apply(pd.Series)], axis=1)
# transfer to long format
df4b=df4.melt(id_vars=['id','patient_id','referring_patient','img_type'], value_name='measure_dataset', var_name='measure_dataset_index').dropna(subset=['measure_dataset'])
df4c = pd.concat([df4b.drop(columns=['measure_dataset']), df4b['measure_dataset'].apply(pd.Series)['data']], axis=1)
df4d = pd.concat([df4c.drop(columns=['data']), df4c['data'].apply(pd.Series)], axis=1)
df4d

,id,upload_user,measure_dataset,referring_patient,patient_id_integrity_check,uuid,img,thumb,img_type,acquisition_date,patient_id,public,created_at,last_modified,image_hash
0,293,"{'id': 10, 'first_name': 'Luisa', 'last_name':...",[],"{'id': 41, 'hasClinicalBaseData': 1, 'complete...",True,a8d73a19-d6a2-40fc-af97-86330452db9e,https://storage.googleapis.com/xray-api-media/...,https://storage.googleapis.com/xray-api-media/...,MRI-LWS-cor,2021-06-08,2590,True,2025-11-07T11:43:37.861522Z,2025-11-07T11:43:38.049791Z,15108edcc38458d55651e740eb6471ab055250408d3795...
1,292,"{'id': 10, 'first_name': 'Luisa', 'last_name':...",[],"{'id': 41, 'hasClinicalBaseData': 1, 'complete...",True,0444cb6e-5cb4-4278-a08a-2fd5e9753e41,https://storage.googleapis.com/xray-api-media/...,https://storage.googleapis.com/xray-api-media/...,Xray-LWS-lat,2020-02-03,2590,True,2025-11-07T11:42:53.887907Z,2025-11-07T11:42:54.077545Z,a3d1f0acdc248e1cfdc775d083a518181a33a5e0e64f68...
2,291,"{'id': 10, 'first_name': 'Luisa', 'last_name':...",[],"{'id': 41, 'hasClinicalBaseData': 1, 'complete...",True,7dd9ccd7-6b39-4aa5-ac11-a8834b11368a,https://storage.googleapis.com/xray-api-media/...,https://storage.googleapis.com/xray-api-media/...,Xray-LWS-ap,2020-02-03,2590,True,2025-11-07T11:42:10.511076Z,2025-11-07T11:42:10.711611Z,68222bc73f7ceac7e8c5506f54a7d15b3f3c598dc49be2...
3,290,"{'id': 10, 'first_name': 'Luisa', 'last_name':...",[],"{'id': 40, 'hasClinicalBaseData': 0, 'complete...",True,7efcf97f-cec1-4e86-9f4c-a166ea91ba32,https://storage.googleapis.com/xray-api-media/...,https://storage.googleapis.com/xray-api-media/...,MRI-LWS-cor,2020-07-13,AW5R,True,2025-11-07T11:33:40.996452Z,2025-11-07T11:33:41.179431Z,9c650b295ca4502de405597fa13e0fd7ca4c9054c906e9...
4,289,"{'id': 10, 'first_name': 'Luisa', 'last_name':...",[],"{'id': 40, 'hasClinicalBaseData': 0, 'complete...",True,e1a3037e-df41-4167-a900-eef85569f462,https://storage.googleapis.com/xray-api-media/...,https://storage.googleapis.com/xray-api-media/...,MRI-LWS-cor,2019-10-23,AW5R,True,2025-11-07T11:32:45.871150Z,2025-11-07T11:32:46.059819Z,a7fee4aae68389bfef181c37a8e12a6c1ba6faf089e34d...


# Analyzing angles data
This section describes the statistical analysis of angles measured by the algorithms inside the xray-annotator app.

In [44]:
df5 = pd.concat([df4d.drop(columns=['xray_measurement_data']), df4d['xray_measurement_data'].apply(pd.Series)], axis=1)
# use df5 for subsequent analysis

df5a = df5.drop(columns=['calibration', 'stateObject','globalParams'])
df5b = pd.concat([df5a.drop(columns=['angles']), df5a['angles'].apply(pd.Series)], axis=1)
df5c=df5b.melt(id_vars=['id','patient_id','referring_patient','img_type','measure_dataset_index'], value_name='angle_data', var_name='angle_index').dropna(subset=['angle_data'])

In [47]:
df6 = pd.concat([df5c.drop(columns=['angle_data']), df5c['angle_data'].apply(pd.Series)], axis=1)
df6b=pd.concat([df6.drop(columns=['cobbAngle','lineColor','lowerLine','upperLine']), df6['cobbAngle'].apply(pd.Series)], axis=1)

In [48]:
# drop all rows where vertebra_lower or vertebra_upper == '?'
df6c = df6b[(df6b['vertebra_lower'] != '?') & (df6b['vertebra_upper'] != '?')]

In [57]:
import math 
spine_labels = ['C0','C1','C2','C3','C4','C5','C6','C7','T1','T2','T3','T4','T5','T6','T7','T8','T9','T10','T11','T12','L1','L2','L3','L4','L5','S1']
spine_regions = {'CT':[0,9],'PT': [10,12],'MT':[13,18],'TL':[19,20],'LU':[21,25]}

# function to map the mean of the indices of vertebra_lower and vertebra_upper to a spine region
def map_to_spine_region(row):
    if row['cobbType']=='Cobb-segmental':
        return row['vertebra_upper']
    lower_index = spine_labels.index(row['vertebra_lower'])
    upper_index = spine_labels.index(row['vertebra_upper'])
    mean_index = math.ceil((lower_index + upper_index) / 2)
    for region, (start, end) in spine_regions.items():
        if start <= mean_index <= end:
            return region
    print(mean_index)
    return 'Unknown'   

df6c.loc[:,'spine_region'] = df6c.apply(map_to_spine_region, axis=1)

In [58]:
df6c.groupby(['id','cobbType','spine_region']).agg({'angle':['mean','count','std']}).reset_index()

id        cobbType spine_region angle          
                                       mean count std
0      5          Spline           LU   8.0     1 NaN
1      5          Spline           TL  23.0     1 NaN
2      6   Cobb-regional           LU  53.0     1 NaN
3      6  Cobb-segmental           L2  11.0     1 NaN
4      6  Cobb-segmental           L3   2.0     1 NaN
..   ...             ...          ...   ...   ...  ..
162  118  Cobb-segmental           L3  10.0     1 NaN
163  118  Cobb-segmental           L4   7.0     1 NaN
164  118  Cobb-segmental           L5   9.0     1 NaN
165  118          Spline           LU  47.0     1 NaN
166  118          Spline           TL  18.0     1 NaN

[167 rows x 6 columns]